# Step 2: Cohort Selection, Outcome Labelling and Feature Engineering (revision 1)

The logic lives in `src/fnpred/cohort.py`. Running `CohortConfig(th_fever=38.5, cancer_flag_mode="ever", lab_mode="same_day", vital_mode="same_day", lab_feature_same_day=True, heme_post_dx_policy="include", event_window_start_day=1, undated_dx_flag=True)`
reproduces the initial-submission dataset exactly (checked on synthetic data).

## Changes from the initial submission
| Item | Initial submission | Revision 1 | Comments |
|---|---|---|---|
| Outcome components | Only `Event` saved | `Event_grade4`, `Event_fn`, `Event_gcsf`, `Event_no_gcsf` (+ `Event_fn_385` for reference) | R1-M1, R2-M2 |
| FN fever threshold | 38.5 °C in code (37.5 °C in the manuscript) | `th_fever = 37.5` (manuscript / JSCO); 38.5 °C kept as `Event_fn_385` | R2-m5 |
| FN timing | Same calendar day (undocumented) | Same calendar day: daily min ANC and daily max temperature (documented) | R2-m5 |
| Exclusions | Silent inner joins | Sequential exclusion with reasons; attrition table; excluded vs. included comparison | R1-M4, R2-M3 |
| Missing values | Described as "no missing values" | Partial missingness (imputed in step 3) reported separately | R1-m6 |
| Cancer-type flags | Whole patient history (incl. future diagnoses) | Only cancer types (ICD-10, 3 characters) with `DIAGNOSIS_DATE_2` on or before each administration date; undated diagnoses are not flagged | R2-m3, R2-m4 |
| Haematological dx after solid tumour | Not examined | Counted; administration days on or after the diagnosis are excluded (`heme_post_dx_policy="exclude_days"`) | R2-m4 |
| Day 1 (DO_DATE) | Same-day labs used both as features and in the outcome window | Same-day (pre-administration) labs stay as features (`lab_feature_same_day=True`); the outcome window starts on **Day 2** (`event_window_start_day=2`); Day-1 events kept as `Y_day1_*` | — |

## Kept from the initial code (authors' decision, 2026-11-25)
- Therapeutic / prophylactic G-CSF is classified **by drug code** (current definition kept).
- The prophylactic G-CSF feature sums doses given on the **2 days** before Day 1 (`pre_gcsf_days`; the manuscript says 3 days).

## Inputs and outputs
- **Input**: `../data/step1_dwh_outputs/step1_all_*.parquet`
- **Output**: `../data/step2_processing_outputs/`
  - `step2_all_tagt_feat`
  - `attrition.csv`
  - `all_chemo_days.parquet`
  - `missingness_included.csv`
  - `heme_after_solid_*`
  - `excluded_vs_included.csv`
  - `outcome_components.csv`


In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath("../src"))
import pandas as pd
from fnpred.cohort import CohortConfig, load_step1, build_cohort, save_cohort, compare_excluded_included
pd.set_option("display.max_columns", 50)

In [ ]:
DATA_DIR   = "../data"
INPUT_DIR  = f"{DATA_DIR}/step1_dwh_outputs"
OUTPUT_DIR = f"{DATA_DIR}/step2_processing_outputs"

cfg = CohortConfig(
    th_fever=37.5,                  
    cancer_flag_mode="as_of_date",  # "ever" reproduces the initial code
    undated_dx_flag=False,          # diagnoses without DIAGNOSIS_DATE_2 are not flagged
    heme_post_dx_policy="exclude_days",  # drop administration days on/after a later haematological diagnosis ("include" reproduces the initial code)
    event_window_start_day=2,       # outcome window Day 2-28 (Day 1 = DO_DATE excluded; initial code: 1)
    lab_mode="per_item",            # latest WBC / ANC taken separately ("same_day" reproduces the initial code)
    vital_mode="per_item",          # latest value of each vital sign taken separately
    lab_feature_same_day=True,      # DO_DATE (pre-administration) labs used as features
)
print(cfg.to_json())

In [ ]:
dfs = load_step1(INPUT_DIR)
res = build_cohort(dfs, cfg)
save_cohort(res, cfg, OUTPUT_DIR)

## Attrition (Fig. 1) and partial missingness

In [ ]:
display(res.attrition)
display(res.missingness)
print(res.all_days["exclusion_reason"].value_counts(dropna=False))

## Check: cancer-type flags use only diagnoses made on or before the administration date (R2-m3)

`DIAGNOSIS_xxx` = 1 only when that cancer type (ICD-10, first 3 characters) has a `DIAGNOSIS_DATE_2`
on or before `DO_DATE`. Cancers diagnosed later, and diagnoses without a date, are not flagged.
Patients whose first diagnosis is a haematological malignancy are excluded as before
(`haematological_malignancy_first_diagnosis`).

In [ ]:
from fnpred.cohort import prepare_cancer
cancer_chk, _ = prepare_cancer(dfs, cfg)
d = res.data
dx_cols = [c for c in d.columns if c.startswith("DIAGNOSIS_")]

# 1) Every flag = 1 must have a dated diagnosis of that type on or before DO_DATE
flags = d[["hs", "DO_DATE"] + dx_cols].melt(id_vars=["hs", "DO_DATE"], var_name="col", value_name="flag")
flags = flags[flags["flag"] == 1].assign(DX3=lambda x: x["col"].str.slice(len("DIAGNOSIS_")))
first_dx = (cancer_chk.dropna(subset=["DX_DATE"]).groupby(["hs", "DX3"])["DX_DATE"].min()
            .rename("FIRST_DX_DATE").reset_index())
flags = flags.merge(first_dx, on=["hs", "DX3"], how="left")
bad = flags[flags["FIRST_DX_DATE"].isna() | (flags["FIRST_DX_DATE"] > flags["DO_DATE"])]
print(f"flags = 1: {len(flags):,}  without a diagnosis on/before DO_DATE: {len(bad):,}")
assert bad.empty, "a cancer-type flag is set before the diagnosis date"

# 2) Diagnoses that are NOT flagged on included days (diagnosed later, or undated)
pairs = d[["hs", "DO_DATE"]].merge(cancer_chk[["hs", "DX3", "DX_DATE"]].drop_duplicates(), on="hs")
dated_before = (pairs.assign(ok=pairs["DX_DATE"] <= pairs["DO_DATE"])
                .groupby(["hs", "DO_DATE", "DX3"])["ok"].any())
undated_types = set(map(tuple, cancer_chk.loc[cancer_chk["DX_DATE"].isna(), ["hs", "DX3"]].values))
not_flagged = dated_before[~dated_before].reset_index()
not_flagged["reason"] = ["undated" if (h, x) in undated_types else "diagnosed after DO_DATE"
                         for h, x in zip(not_flagged["hs"], not_flagged["DX3"])]
# a type that has both an undated and a later-dated record is shown as "undated"
print("\n(administration day, cancer type) pairs not flagged:")
display(not_flagged.groupby("reason").agg(day_type_pairs=("hs", "size"), patients=("hs", "nunique")))
display(not_flagged.groupby("DX3").agg(day_type_pairs=("hs", "size"), patients=("hs", "nunique"))
        .sort_values("day_type_pairs", ascending=False).head(15))

# Undated registrations in the cancer table
und = cancer_chk[cancer_chk["DX_DATE"].isna()]
print(f"\nregistrations without DIAGNOSIS_DATE_2: {len(und):,} rows, {und['hs'].nunique():,} patients "
      f"({und['hs'].isin(set(d['hs'])).sum():,} rows belong to included patients)")

## Excluded vs. included patients (R1-M4, R2-M3)

In [ ]:
cmp = compare_excluded_included(res.all_days)
display(cmp); cmp.to_csv(f"{OUTPUT_DIR}/excluded_vs_included.csv", index=False)

## Outcome components (R2-M2)

In [ ]:
d = res.data
comp = ["Event", "Event_grade4", "Event_fn", "Event_fn_385", "Event_gcsf", "Event_no_gcsf"]
tab = pd.DataFrame({"administration_days": d[comp].sum(), "proportion": d[comp].mean(),
                    "patients_with_>=1": d.groupby("hs")[comp].max().sum()})
# G-CSF-only events (no Grade 4 / FN recorded in the window)
gonly = (d["Event_gcsf"] == 1) & (d["Event_no_gcsf"] == 0)
tab.loc["G-CSF only"] = [gonly.sum(), gonly.mean(), d.loc[gonly, "hs"].nunique()]
display(tab); tab.to_csv(f"{OUTPUT_DIR}/outcome_components.csv")
print("administration days without any ANC/WBC in the 28-day window:", (d["Y_fu_lab_days"] == 0).mean().round(3))

## Haematological diagnoses after a solid tumour (R2-m4)

In [ ]:
print(json.dumps({k: v for k, v in res.heme.items() if k != "top_drugs_after_heme_dx"}, indent=1, ensure_ascii=False))
display(res.heme["top_drugs_after_heme_dx"])

## Day 1 (DO_DATE) and the outcome window
The outcome window starts on Day 2. `Y_day1_*` = the event was already present on Day 1, so it is not counted in the revised outcome.


In [ ]:
d = res.data
day1 = d[["Y_day1_grade4", "Y_day1_fn", "Y_day1_gcsf"]]
print("administration days with an event on Day 1:", day1.sum().to_dict())
print("... of which have no event on Day 2-28 (removed from the outcome):",
      int(((day1.max(axis=1) == 1) & (d["Event"] == 0)).sum()))
print("lab features measured on DO_DATE (days since = 0):",
      {c: f"{(d[c] == 0).mean():.1%}" for c in d.columns if c in ("daysLastWBC", "daysLastNEUT#")})


## Reference: initial-submission settings (for the response letter)

In [ ]:
legacy_cfg = CohortConfig(th_fever=38.5, cancer_flag_mode="ever", lab_mode="same_day", vital_mode="same_day",
                          lab_feature_same_day=True, heme_post_dx_policy="include",
                          event_window_start_day=1, undated_dx_flag=True)
legacy = build_cohort(dfs, legacy_cfg, verbose=False)
print("initial-submission settings: patients", legacy.data["hs"].nunique(), "days", len(legacy.data),
      "event", round(legacy.data["Event"].mean(), 4))
print("revision settings:           patients", res.data["hs"].nunique(), "days", len(res.data),
      "event", round(res.data["Event"].mean(), 4))